# DiffSplat — Google Colab Notebook

**Text-to-3D and Image-to-3D generation with Gaussian Splatting.**

This notebook runs the DiffSplat inference pipeline. To avoid re-downloading and re-installing large packages on every session, it uses **Google Drive as a persistent cache** for:
- pip wheel downloads
- Model checkpoints
- The DiffSplat repository clone

### Requirements
- A Google account with Google Drive (needs ~15 GB free for SD1.5 checkpoints)
- A GPU runtime: `Runtime > Change runtime type > GPU`

### Workflow
1. Mount Drive & configure paths
2. Install dependencies (cached to Drive)
3. Clone DiffSplat repo (cached to Drive)
4. Download model checkpoints (cached to Drive)
5. Run Text-to-3D or Image-to-3D inference
6. View outputs

## Step 0 — Verify GPU

Make sure you selected a GPU runtime before continuing.

In [ ]:
import subprocess, sys, os

try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                            capture_output=True, text=True, check=True)
    print("GPU detected:", result.stdout.strip())
except (subprocess.CalledProcessError, FileNotFoundError):
    raise RuntimeError(
        "No GPU found. Go to Runtime > Change runtime type > Hardware accelerator > GPU."
    )

# Confirm we are in Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running inside Google Colab.")
except ImportError:
    IN_COLAB = False
    print("Not running inside Colab — Drive mount will be skipped.")

# Check NumPy version — NumPy 2.x breaks binary compatibility with Colab's
# pre-compiled PyTorch and CUDA extensions (dtype struct size mismatch).
import numpy as np
np_major = int(np.__version__.split(".")[0])
if np_major >= 2:
    raise RuntimeError(
        f"NumPy {np.__version__} detected (2.x is incompatible with this runtime).\n"
        "Run Step 2 (Install Dependencies) first, then go to\n"
        "Runtime > Restart session and re-run from this cell."
    )
print(f"NumPy {np.__version__} OK.")

## Step 1 — Mount Google Drive & Configure Paths

All large assets (packages, checkpoints, repo) are stored under `DRIVE_BASE` so they survive runtime restarts.

In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive/DiffSplat")
else:
    DRIVE_BASE = Path.home() / "DiffSplat_local_cache"

# Sub-directories inside Drive
PIP_CACHE     = DRIVE_BASE / "pip_cache"       # pip wheel cache (avoids re-downloading)
REPO_DIR      = DRIVE_BASE / "DiffSplat"       # official DiffSplat repo clone
WRAPPER_DIR   = DRIVE_BASE / "diffsplat-wrapper" # this wrapper repo
CHECKPOINTS   = DRIVE_BASE / "checkpoints"     # downloaded model weights
DATA_DIR      = DRIVE_BASE / "data"            # datasets (T3Bench, GSO)
OUT_DIR       = DRIVE_BASE / "outputs"         # inference outputs
WHEELS_DIR    = DRIVE_BASE / "built_wheels"    # pre-compiled extension wheels
LOGS_DIR      = DRIVE_BASE / "logs"

# Create all directories
for d in [PIP_CACHE, CHECKPOINTS, DATA_DIR, OUT_DIR, WHEELS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Drive base:", DRIVE_BASE)
print("Pip cache: ", PIP_CACHE)
print("Repo dir:  ", REPO_DIR)
print("Outputs:   ", OUT_DIR)

## Step 2 — Install Dependencies

Packages are installed into the Colab runtime but **wheels are cached on Drive** so subsequent sessions download nothing. Compiled extensions (RaDe-GS rasterizer, custom diffusers) are built once per CUDA version and the resulting wheel is saved to Drive.

The first run takes ~10–15 min. Subsequent sessions take ~2–3 min (wheel install only).

In [ ]:
import subprocess, sys, os
from pathlib import Path

def pip(*args, cache_dir=None, quiet=False):
    """Run pip install, optionally using Drive as the wheel cache."""
    cmd = [sys.executable, "-m", "pip", "install"]
    if cache_dir:
        cmd += ["--cache-dir", str(cache_dir)]
    if quiet:
        cmd.append("-q")
    cmd += list(args)
    print("Running:", " ".join(str(a) for a in cmd))
    result = subprocess.run(cmd, capture_output=False)
    if result.returncode != 0:
        raise RuntimeError(f"pip install failed for: {args}")

def get_cuda_python_tag():
    cuda = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
    cuda_ver = "unknown"
    for line in cuda.stdout.splitlines():
        if "release" in line:
            cuda_ver = line.split("release")[-1].strip().split(",")[0].strip().replace(".", "")
            break
    py_ver = f"{sys.version_info.major}{sys.version_info.minor}"
    return f"cu{cuda_ver}_py{py_ver}"

RUNTIME_TAG = get_cuda_python_tag()
print("Runtime tag:", RUNTIME_TAG)

INSTALL_MARKER = DRIVE_BASE / f".packages_installed_{RUNTIME_TAG}"

if INSTALL_MARKER.exists():
    print(f"Packages already installed for {RUNTIME_TAG} (marker found). Skipping.")
    print("Delete", INSTALL_MARKER, "to force reinstall.")
else:
    print("Installing packages (wheels cached to Drive)...")

    pip("--upgrade", "pip", "setuptools", "wheel", cache_dir=PIP_CACHE, quiet=True)

    # Pin diffusers==0.32 (official DiffSplat requirement) and
    # transformers<4.47 — transformers 4.47+ removed FLAX_WEIGHTS_NAME
    # which diffusers 0.32 and the diffusers_diffsplat extension depend on.
    pip("diffusers==0.32",
        "transformers>=4.44,<4.47",
        "huggingface_hub", "scikit-image", "gpustat",
        "einops", "omegaconf", "accelerate",
        "timm", "lpips", "rembg[gpu]",
        "ninja",          # required for CUDA extension builds (RaDe-GS rasterizer)
        cache_dir=PIP_CACHE)

    # xformers: auto-match the installed PyTorch + CUDA version
    try:
        import torch
        cuda_tag = "cu" + torch.version.cuda.replace(".", "")[:3]   # e.g. "cu128"
        print(f"Detected PyTorch {torch.__version__}, CUDA tag: {cuda_tag}")
        index_url = f"https://download.pytorch.org/whl/{cuda_tag}"
        pip("xformers", "--index-url", index_url, cache_dir=PIP_CACHE)
    except Exception as e:
        print(f"xformers install skipped or failed: {e}")

    # Pin NumPy to 1.x — scikit-image/rembg can pull in NumPy 2.x which
    # breaks binary compatibility (dtype struct 88 vs 96 bytes).
    pip("numpy<2.0", cache_dir=PIP_CACHE)

    INSTALL_MARKER.touch()
    print("\nCore packages installed.")
    print("\n*** IMPORTANT: Runtime > Restart session, then re-run from Step 0. ***")

## Step 3 — Clone / Update Repositories

In [ ]:
import subprocess, sys, shutil
from pathlib import Path

OFFICIAL_REPO_URL = "https://github.com/chenguolin/DiffSplat.git"
WRAPPER_REPO_URL  = "https://github.com/kaustubh484/diffsplat.git"

def is_valid_git_repo(path: Path) -> bool:
    result = subprocess.run(
        ["git", "-C", str(path), "rev-parse", "--git-dir"],
        capture_output=True
    )
    return result.returncode == 0

def git_clone_or_update(url: str, target: Path, depth: int = 1):
    """Clone to local /tmp/ first (fast POSIX FS), then move to Drive.
    Drive FUSE has high latency for many small writes (e.g. .git objects);
    cloning directly there often times out or leaves a broken partial clone."""

    if target.exists():
        if is_valid_git_repo(target):
            print(f"Repo already exists at {target}, pulling latest...")
            result = subprocess.run(
                ["git", "-C", str(target), "pull", "--ff-only"],
                capture_output=True, text=True
            )
            print(result.stdout.strip() or "Already up to date.")
            if result.returncode != 0:
                print(f"Pull failed: {result.stderr.strip()}")
            return
        else:
            print(f"Found broken/incomplete clone at {target} — removing and re-cloning...")
            shutil.rmtree(target)

    # Clone to /tmp/ first (local FS, no FUSE overhead), then move to Drive
    tmp_target = Path("/tmp") / target.name
    if tmp_target.exists():
        shutil.rmtree(tmp_target)

    print(f"Cloning {url} -> {tmp_target} (local, then moving to Drive)...")
    subprocess.run(
        ["git", "clone", "--depth", str(depth), url, str(tmp_target)],
        check=True
    )

    print(f"Moving to Drive: {tmp_target} -> {target}")
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(tmp_target), str(target))
    print("Done.")

# Clone official DiffSplat repo (contains model code, inference scripts, configs)
git_clone_or_update(OFFICIAL_REPO_URL, REPO_DIR)

# Clone the wrapper repo (contains diffsplat_tools package)
git_clone_or_update(WRAPPER_REPO_URL, WRAPPER_DIR)

print("\nRepo dirs:")
print(" Official:", REPO_DIR)
print(" Wrapper: ", WRAPPER_DIR)

## Step 4 — Install the Official DiffSplat Requirements

The official repo ships its own `settings/requirements.txt`. We install those here, using Drive as the pip cache.

In [ ]:
import subprocess, sys

req_file = REPO_DIR / "settings" / "requirements.txt"
if not req_file.exists():
    print(f"requirements.txt not found at {req_file}. Skipping.")
else:
    REQS_MARKER = DRIVE_BASE / f".reqs_installed_{RUNTIME_TAG}"
    if REQS_MARKER.exists():
        print("Official requirements already installed. Skipping.")
    else:
        print("Installing official DiffSplat requirements...")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "--cache-dir", str(PIP_CACHE),
             "-r", str(req_file)],
            check=True
        )
        REQS_MARKER.touch()
        print("Done.")

## Step 5 — Build Compiled Extensions

Two compiled packages are needed:
1. **RaDe-GS Gaussian rasterizer** — builds a CUDA extension
2. **diffusers_diffsplat** — custom diffusers fork (pure Python editable install)

Built wheels are stored on Drive so subsequent sessions just `pip install` from the cached wheel (no recompilation).

In [ ]:
import subprocess, sys, shutil, os
from pathlib import Path

# --- diffusers_diffsplat ---
REPO_DIR_STR = str(REPO_DIR)
if REPO_DIR_STR not in sys.path:
    sys.path.insert(0, REPO_DIR_STR)
    print(f"Added repo root to sys.path: {REPO_DIR_STR}")
else:
    print("Repo root already on sys.path.")

# --- Patch mv_attention_processor.py (missing typing imports) ---
# extensions/diffusers_diffsplat/models/mv_attention_processor.py uses
# Optional, Callable, etc. from typing but forgets to import them.
mv_attn_path = REPO_DIR / "extensions" / "diffusers_diffsplat" / "models" / "mv_attention_processor.py"
if mv_attn_path.exists():
    mv_content = mv_attn_path.read_text()
    needs_typing = "from typing import" not in mv_content and (
        "Optional" in mv_content or "Callable" in mv_content or "Tuple" in mv_content
    )
    if needs_typing:
        mv_content = "from typing import Callable, Optional, Tuple\n" + mv_content
        mv_attn_path.write_text(mv_content)
        print("mv_attention_processor.py: added missing typing imports")
    else:
        print("mv_attention_processor.py: typing imports already present, skipping.")
else:
    print(f"WARNING: mv_attention_processor.py not found at {mv_attn_path}")

try:
    from extensions.diffusers_diffsplat import (  # noqa: F401
        UNetMV2DConditionModel, StableMVDiffusionPipeline,
    )
    print("extensions.diffusers_diffsplat imported OK.")
except ImportError as e:
    print(f"WARNING: import failed: {e}")
    if "FLAX_WEIGHTS_NAME" in str(e):
        print(f"  \u2192 Delete {INSTALL_MARKER} and re-run Step 2 to pin transformers<4.47.")
    else:
        print("  \u2192 Make sure the official DiffSplat repo was cloned in Step 3.")

# --- RaDe-GS Gaussian rasterizer ---
RADE_GS_DIR          = REPO_DIR / "extensions" / "RaDe-GS"
RASTERIZER_SUBMODULE = RADE_GS_DIR / "submodules" / "diff-gaussian-rasterization"
existing_wheels      = list(WHEELS_DIR.glob("diff_gaussian_rasterization*.whl"))
RASTERIZER_MARKER    = DRIVE_BASE / f".rasterizer_installed_{RUNTIME_TAG}"

if RASTERIZER_MARKER.exists():
    print("RaDe-GS rasterizer already installed for this runtime.")
elif existing_wheels:
    wheel = existing_wheels[0]
    print(f"Installing rasterizer from cached wheel: {wheel.name}")
    subprocess.run([sys.executable, "-m", "pip", "install", str(wheel)], check=True)
    RASTERIZER_MARKER.touch()
    print("Done.")
else:
    print("Cloning RaDe-GS and building rasterizer...")
    RADE_GS_DIR.parent.mkdir(parents=True, exist_ok=True)

    if not RADE_GS_DIR.exists():
        tmp_rade = Path("/tmp/RaDe-GS")
        if tmp_rade.exists():
            shutil.rmtree(tmp_rade)
        subprocess.run(
            ["git", "clone", "https://github.com/BaowenZ/RaDe-GS.git",
             "--recursive", str(tmp_rade)],
            check=True
        )
        shutil.move(str(tmp_rade), str(RADE_GS_DIR))

    subprocess.run(
        ["git", "submodule", "update", "--init", "--recursive"],
        cwd=RADE_GS_DIR, check=True
    )

    if not RASTERIZER_SUBMODULE.exists():
        raise RuntimeError(
            f"Submodule not found at {RASTERIZER_SUBMODULE}\n"
            "Delete the RaDe-GS directory on Drive and re-run this cell."
        )

    import torch
    compute_caps = set()
    for i in range(torch.cuda.device_count()):
        major, minor = torch.cuda.get_device_capability(i)
        compute_caps.add(f"{major}.{minor}")
    arch_list = " ".join(sorted(compute_caps)) or "7.5 8.0 8.6 9.0"
    cuda_bin = "/usr/local/cuda/bin"
    current_path = os.environ.get("PATH", "")
    build_env = {
        **os.environ,
        "TORCH_CUDA_ARCH_LIST": arch_list,
        "PATH": f"{cuda_bin}:{current_path}" if cuda_bin not in current_path else current_path,
    }
    print(f"Building for CUDA arch: {arch_list}")

    nvcc_check = subprocess.run(["nvcc", "--version"], capture_output=True, text=True, env=build_env)
    if nvcc_check.returncode != 0:
        raise RuntimeError("nvcc not found. Run:  !apt-get install -y cuda-toolkit")
    print("nvcc:", nvcc_check.stdout.splitlines()[3])

    tmp_src = Path("/tmp/diff-gaussian-rasterization")
    if tmp_src.exists():
        shutil.rmtree(tmp_src)
    print("Copying rasterizer source to /tmp/...")
    shutil.copytree(str(RASTERIZER_SUBMODULE), str(tmp_src))

    build_dir = Path("/tmp/rasterizer_build")
    if build_dir.exists():
        shutil.rmtree(build_dir)
    build_dir.mkdir()

    print("Building wheel (this takes ~5-10 min)...")
    proc = subprocess.Popen(
        [sys.executable, "-m", "pip", "wheel",
         "--no-build-isolation",
         "--wheel-dir", str(build_dir),
         str(tmp_src)],
        env=build_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in iter(proc.stdout.readline, ""):
        print(line, end="", flush=True)
    build_rc = proc.wait()

    if build_rc != 0:
        raise RuntimeError("Rasterizer build failed — see compiler output above.")

    built = list(build_dir.glob("diff_gaussian_rasterization*.whl"))
    if not built:
        raise RuntimeError("Build succeeded but no .whl file found.")

    saved = WHEELS_DIR / built[0].name
    shutil.copy2(built[0], saved)
    print(f"Wheel saved to Drive: {saved.name}")
    subprocess.run([sys.executable, "-m", "pip", "install", str(saved)], check=True)
    RASTERIZER_MARKER.touch()
    print("RaDe-GS rasterizer installed OK.")

# --- Patch gs_util.py for RaDe-GS rasterizer compatibility ---
# DiffSplat was written for a custom rasterizer fork that:
#   1. Accepts require_coord in GaussianRasterizationSettings
#   2. Returns 8 values: (image, radii, coord, mcoord, depth, mdepth, alpha, normal)
#
# The standard RaDe-GS rasterizer returns 6 values:
#   (color, radii, depth, alpha, normal, depth_distortion)
#
# Mapping — values actually used (image, depth, alpha, normal):
#   DiffSplat positions:  0       4       6      7
#   RaDe-GS positions:    0       2       3      4
#
# Patch 1: remove require_coord= line (no such field in the NamedTuple)
# Patch 2: rewrite the unpack line to match the 6-value RaDe-GS return
gs_util_path = REPO_DIR / "src" / "models" / "gs_render" / "gs_util.py"
if gs_util_path.exists():
    content = gs_util_path.read_text()
    changed = False

    # Patch 1: remove require_coord= argument
    if "require_coord=" in content:
        lines = content.splitlines(keepends=True)
        content = "".join(l for l in lines if "require_coord=" not in l)
        changed = True
        print("gs_util.py patch 1: removed require_coord= argument")

    # Patch 2: rewrite 8-value unpack to 6-value RaDe-GS unpack
    OLD_UNPACK = "image, _, _, _, depth, _, alpha, normal = rasterizer("
    NEW_UNPACK = "image, _, depth, alpha, normal, _ = rasterizer(  # patched: RaDe-GS returns 6 values"
    if OLD_UNPACK in content:
        content = content.replace(OLD_UNPACK, NEW_UNPACK)
        changed = True
        print("gs_util.py patch 2: rewrote rasterizer unpack for 6-value RaDe-GS output")
    elif NEW_UNPACK in content:
        print("gs_util.py patch 2: already applied.")
    else:
        print("WARNING: unpack line not found — gs_util.py may have changed upstream.")

    if changed:
        gs_util_path.write_text(content)
else:
    print(f"WARNING: gs_util.py not found at {gs_util_path}")

print("\nExtension setup complete.")


## Step 6 — Add Wrapper to Python Path

This makes the `diffsplat_tools` package importable from the wrapper repo.

In [ ]:
import sys, os
from pathlib import Path

# Wrapper repo (diffsplat_tools package)
wrapper_str = str(WRAPPER_DIR)
if wrapper_str not in sys.path:
    sys.path.insert(0, wrapper_str)
    print(f"Added to sys.path: {wrapper_str}")
else:
    print("Wrapper already on sys.path.")

# Official repo root — inference script uses "from extensions.diffusers_diffsplat import ..."
repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
    print(f"Added to sys.path: {repo_str}")
else:
    print("Repo root already on sys.path.")

# Build PYTHONPATH string for subprocesses (child processes don't inherit sys.path changes)
existing_pypath = os.environ.get("PYTHONPATH", "")
new_paths = ":".join(p for p in [wrapper_str, repo_str] if p not in existing_pypath)
if new_paths:
    os.environ["PYTHONPATH"] = new_paths + (":" + existing_pypath if existing_pypath else "")
    print(f"PYTHONPATH set to: {os.environ['PYTHONPATH']}")

# Sanity checks
try:
    from diffsplat_tools.constants import MODEL_SPECS
    print("diffsplat_tools imported OK. Available models:", list(MODEL_SPECS.keys()))
except ImportError as e:
    print(f"diffsplat_tools import failed: {e}")

try:
    from extensions.diffusers_diffsplat import UNetMV2DConditionModel  # noqa: F401
    print("extensions.diffusers_diffsplat imported OK.")
except ImportError as e:
    print(f"extensions.diffusers_diffsplat import failed: {e}")
    print("Re-run Step 5 to ensure the repo root is on the path.")

## Step 7 — Download Model Checkpoints

Checkpoints are large (~2–10 GB). They are downloaded once to Drive and reused across sessions.

Choose your model:
- `sd15` — Stable Diffusion 1.5 (recommended for T4 GPUs, ~2 GB)
- `pas` — PAS variant (fp16, ~2 GB)
- `sd35m` — Stable Diffusion 3.5 Medium (needs more VRAM, ~5 GB)

In [ ]:
# @title Model Selection { run: "auto" }
MODEL = "sd15"  # @param ["sd15", "pas", "sd35m"]
VARIANT = "text"  # @param ["text", "image", "both"]

print(f"Selected model: {MODEL}, variant: {VARIANT}")

In [ ]:
import subprocess, sys, os
from diffsplat_tools.constants import MODEL_SPECS

spec = MODEL_SPECS[MODEL]

# Environment for HuggingFace (use Drive as HF cache)
HF_HOME = str(DRIVE_BASE / "hf_home")
Path(HF_HOME).mkdir(parents=True, exist_ok=True)

env = {**os.environ, "HF_HOME": HF_HOME}

CHECKPOINTS.mkdir(parents=True, exist_ok=True)

base_cmd = [
    sys.executable,
    "download_ckpt.py",
    "--local_dir", str(CHECKPOINTS),
    "--model_type", spec.model_type,
]

def checkpoint_exists(tag: str) -> bool:
    return (CHECKPOINTS / tag).exists() and any((CHECKPOINTS / tag).iterdir())

if VARIANT in {"text", "both"}:
    if checkpoint_exists(spec.text_tag):
        print(f"Text checkpoint already downloaded: {spec.text_tag}")
    else:
        print(f"Downloading text checkpoint: {spec.text_tag}...")
        subprocess.run(base_cmd, cwd=REPO_DIR, env=env, check=True)
        print("Done.")

if VARIANT in {"image", "both"}:
    if checkpoint_exists(spec.image_tag):
        print(f"Image checkpoint already downloaded: {spec.image_tag}")
    else:
        print(f"Downloading image checkpoint: {spec.image_tag}...")
        subprocess.run([*base_cmd, "--image_cond"], cwd=REPO_DIR, env=env, check=True)
        print("Done.")

## Step 8 — Text-to-3D Inference

Generate a 3D Gaussian splat from a text prompt.

In [ ]:
# @title Text-to-3D settings
PROMPT           = "a toy robot"       # @param {type:"string"}
SEED             = 42                  # @param {type:"integer"}
NUM_STEPS        = 50                  # @param {type:"integer"}
GUIDANCE_SCALE   = 7.5                 # @param {type:"number"}
OUTPUT_VIDEO     = "gif"               # @param ["gif", "mp4", "none"]
HALF_PRECISION   = True                # @param {type:"boolean"}

In [ ]:
import subprocess, sys, os
from pathlib import Path
from diffsplat_tools.constants import MODEL_SPECS

spec = MODEL_SPECS[MODEL]

env = {
    **os.environ,
    "HF_HOME":    str(DRIVE_BASE / "hf_home"),
    "TORCH_HOME": str(DRIVE_BASE / "torch_home"),
    "PYTHONPATH": str(REPO_DIR) + ":" + os.environ.get("PYTHONPATH", ""),
}

cmd = [
    sys.executable, spec.script,
    "--config_file",         spec.config,
    "--tag",                 spec.text_tag,
    "--output_dir",          str(CHECKPOINTS),
    "--seed",                str(SEED),
    "--gpu_id",              "0",
    "--allow_tf32",
    "--prompt",              PROMPT,
    "--num_inference_steps", str(NUM_STEPS),
    "--guidance_scale",      str(GUIDANCE_SCALE),
]

if HALF_PRECISION:
    cmd.append("--half_precision")
if OUTPUT_VIDEO and OUTPUT_VIDEO != "none":
    cmd += ["--output_video_type", OUTPUT_VIDEO]

print("Running inference...")
print(" ".join(str(c) for c in cmd))
print(f"\nOutputs will be written to: {CHECKPOINTS / spec.text_tag}\n")

# Stream stdout+stderr line-by-line so Jupyter shows output in real time.
# subprocess.run() without capture_output is often silently buffered in Colab.
proc = subprocess.Popen(
    cmd, cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
returncode = proc.wait()

if returncode == 0:
    print(f"\nInference complete. Outputs in:\n  {CHECKPOINTS / spec.text_tag}")
else:
    print(f"\nInference failed (exit code {returncode}).")

## Step 9 — Image-to-3D Inference

Reconstruct a 3D Gaussian splat from a single input image.

Either upload an image or provide a URL below. The image checkpoint must be downloaded (set `VARIANT = "image"` or `"both"` in Step 7).

In [ ]:
# @title Image-to-3D settings
IMAGE_SOURCE     = "upload"            # @param ["upload", "path"]
IMAGE_PATH       = ""                  # @param {type:"string"} path on Drive or /content/
IMG_PROMPT       = ""                  # @param {type:"string"} optional text hint
ELEVATION        = 0.0                 # @param {type:"number"} camera elevation in degrees
REMBG            = True                # @param {type:"boolean"} remove background first
IMG_SEED         = 42                  # @param {type:"integer"}
IMG_STEPS        = 50                  # @param {type:"integer"}
IMG_GUIDANCE     = 5.0                 # @param {type:"number"}
IMG_VIDEO        = "gif"               # @param ["gif", "mp4", "none"]
HALF_PRECISION   = True                # @param {type:"boolean"}

In [ ]:
from pathlib import Path

if IMAGE_SOURCE == "upload":
    if IN_COLAB:
        from google.colab import files
        print("Upload your image file:")
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No file uploaded.")
        img_name = list(uploaded.keys())[0]
        img_path = Path("/content") / img_name
        print(f"Uploaded: {img_path}")
    else:
        raise ValueError("Upload only works inside Colab. Set IMAGE_SOURCE='path' and provide IMAGE_PATH.")
else:
    if not IMAGE_PATH:
        raise ValueError("Set IMAGE_PATH to the path of your input image.")
    img_path = Path(IMAGE_PATH).expanduser()
    if not img_path.exists():
        raise FileNotFoundError(f"Image not found: {img_path}")
    print(f"Using image: {img_path}")

# Show the input image
from IPython.display import Image as IPyImage, display
display(IPyImage(filename=str(img_path), width=256))

In [ ]:
import subprocess, sys, os
from pathlib import Path
from diffsplat_tools.constants import MODEL_SPECS

spec = MODEL_SPECS[MODEL]

# Guard: check the image checkpoint was actually downloaded
ckpt_dir = CHECKPOINTS / spec.image_tag
if not ckpt_dir.exists() or not any(ckpt_dir.iterdir()):
    raise RuntimeError(
        f"Image checkpoint not found: {ckpt_dir}\n\n"
        "To fix:\n"
        "  1. Go to the Model Selection cell (Step 7)\n"
        "  2. Set  VARIANT = \"image\"  (or \"both\")\n"
        "  3. Re-run Step 7 then Step 4 (Download Checkpoints)\n"
        "  4. Come back and re-run this cell."
    )

env = {
    **os.environ,
    "HF_HOME":    str(DRIVE_BASE / "hf_home"),
    "TORCH_HOME": str(DRIVE_BASE / "torch_home"),
    "PYTHONPATH": str(REPO_DIR) + ":" + os.environ.get("PYTHONPATH", ""),
}

cmd = [
    sys.executable, spec.script,
    "--config_file",         spec.config,
    "--tag",                 spec.image_tag,
    "--output_dir",          str(CHECKPOINTS),
    "--seed",                str(IMG_SEED),
    "--gpu_id",              "0",
    "--allow_tf32",
    "--image_path",          str(img_path),
    "--elevation",           str(ELEVATION),
    "--num_inference_steps", str(IMG_STEPS),
    "--guidance_scale",      str(IMG_GUIDANCE),
]

if IMG_PROMPT:
    cmd += ["--prompt", IMG_PROMPT]
if REMBG:
    cmd.append("--rembg_and_center")
if HALF_PRECISION:
    cmd.append("--half_precision")
if IMG_VIDEO and IMG_VIDEO != "none":
    cmd += ["--output_video_type", IMG_VIDEO]

print("Running image-to-3D inference...")
print(" ".join(str(c) for c in cmd))
print(f"\nOutputs will be written to: {CHECKPOINTS / spec.image_tag}\n")

proc = subprocess.Popen(
    cmd, cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
returncode = proc.wait()

if returncode == 0:
    print(f"\nDone. Outputs in:\n  {CHECKPOINTS / spec.image_tag}")
else:
    print(f"\nInference failed (exit code {returncode}).")


## Step 10 — View Outputs

Display rendered images and videos from the most recent inference run.

In [ ]:
# @title Choose which output to view
VIEW_MODE = "text"  # @param ["text", "image"]
MAX_IMAGES = 8      # @param {type:"integer"}

In [ ]:
from pathlib import Path
from IPython.display import display, Image as IPyImage
from diffsplat_tools.constants import MODEL_SPECS

spec = MODEL_SPECS[MODEL]
tag = spec.text_tag if VIEW_MODE == "text" else spec.image_tag
view_dir = CHECKPOINTS / tag

if not view_dir.exists():
    print(f"No outputs found in {view_dir}. Run inference first.")
else:
    IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}
    images = sorted(
        [p for p in view_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTS
         and "checkpoints" not in str(p)],   # skip checkpoint weight files
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )[:MAX_IMAGES]

    if images:
        print(f"Showing {len(images)} most recent images from {view_dir}:")
        for img_p in images:
            print(f"  {img_p.relative_to(CHECKPOINTS)}")
            display(IPyImage(filename=str(img_p), width=320))
    else:
        print(f"No output images found in {view_dir}.")

    gifs = sorted(
        [p for p in view_dir.rglob("*.gif")
         if "checkpoints" not in str(p)],
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )[:3]
    if gifs:
        print(f"\nShowing {len(gifs)} GIF(s):")
        for gif_p in gifs:
            print(f"  {gif_p.relative_to(CHECKPOINTS)}")
            display(IPyImage(filename=str(gif_p), width=320))

## Ablation Study — Effect of Base Diffusion Prior

Compares **sd15** (Stable Diffusion 1.5), **pas** (PixArt-Sigma FP16), and **sd35m** (Stable Diffusion 3.5 Medium) on the same 5 prompts with seed 42.  
The goal is to test whether a stronger 2D diffusion prior produces higher-quality 3D outputs.

Run the four cells below in order. Each step skips work that is already done.

In [ ]:
# @title Ablation — Step 1: Download All Three Checkpoints
import subprocess, sys, os
from pathlib import Path
from diffsplat_tools.constants import MODEL_SPECS

ABLATION_MODELS = ["sd15", "pas", "sd35m"]

HF_HOME = str(DRIVE_BASE / "hf_home")
Path(HF_HOME).mkdir(parents=True, exist_ok=True)
env = {**os.environ, "HF_HOME": HF_HOME}

def checkpoint_exists(tag: str) -> bool:
    d = CHECKPOINTS / tag
    return d.exists() and any(d.iterdir())

for model_name in ABLATION_MODELS:
    spec = MODEL_SPECS[model_name]
    if checkpoint_exists(spec.text_tag):
        print(f"[{model_name}] already present: {spec.text_tag}")
        continue
    print(f"[{model_name}] Downloading text checkpoint: {spec.text_tag} ...")
    base_cmd = [
        sys.executable, "download_ckpt.py",
        "--local_dir", str(CHECKPOINTS),
        "--model_type", spec.model_type,
    ]
    proc = subprocess.Popen(
        base_cmd, cwd=REPO_DIR, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in iter(proc.stdout.readline, ""):
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Checkpoint download failed for {model_name} (exit {rc})")
    print(f"[{model_name}] Done.")

print("\nAll ablation checkpoints ready.")


In [ ]:
# @title Ablation — Step 2: Run Inference (3 models × 5 prompts)
import subprocess, sys, os, shutil, time
from pathlib import Path
from diffsplat_tools.constants import MODEL_SPECS

ABLATION_PROMPTS = [
    "a toy robot",
    "a red sports car",
    "a wooden chair",
    "a tropical plant in a pot",
    "a birthday cake with candles",
]
ABLATION_SEED     = 42
ABLATION_STEPS    = 50
ABLATION_GUIDANCE = 7.5
ABLATION_OUT      = DRIVE_BASE / "ablation_outputs"   # separate from normal outputs
ABLATION_OUT.mkdir(parents=True, exist_ok=True)

env = {
    **os.environ,
    "HF_HOME":    str(DRIVE_BASE / "hf_home"),
    "TORCH_HOME": str(DRIVE_BASE / "torch_home"),
    "PYTHONPATH": str(REPO_DIR) + ":" + os.environ.get("PYTHONPATH", ""),
}

def prompt_slug(p: str) -> str:
    return p.replace(" ", "_")[:40]

for model_name in ["sd15", "pas", "sd35m"]:
    spec = MODEL_SPECS[model_name]
    for prompt in ABLATION_PROMPTS:
        slug = prompt_slug(prompt)
        dest = ABLATION_OUT / model_name / slug
        if dest.exists() and list(dest.glob("*.gif")):
            print(f"[{model_name}] '{prompt}' already done — skipping.")
            continue

        print(f"\n{'='*60}")
        print(f"[{model_name}] '{prompt}'")
        print('='*60)

        # Snapshot existing GIFs so we know which ones are new after the run
        tag_dir  = CHECKPOINTS / spec.text_tag
        tag_dir.mkdir(parents=True, exist_ok=True)
        before   = set(tag_dir.rglob("*.gif"))
        t_start  = time.time()

        cmd = [
            sys.executable, spec.script,
            "--config_file",         spec.config,
            "--tag",                 spec.text_tag,
            "--output_dir",          str(CHECKPOINTS),
            "--seed",                str(ABLATION_SEED),
            "--gpu_id",              "0",
            "--allow_tf32",
            "--prompt",              prompt,
            "--num_inference_steps", str(ABLATION_STEPS),
            "--guidance_scale",      str(ABLATION_GUIDANCE),
            "--output_video_type",   "gif",
            "--half_precision",
        ]
        proc = subprocess.Popen(
            cmd, cwd=REPO_DIR, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in iter(proc.stdout.readline, ""):
            print(line, end="", flush=True)
        rc = proc.wait()
        elapsed = time.time() - t_start

        if rc != 0:
            print(f"  WARNING: inference failed (exit {rc}) — continuing.")
            continue

        # Collect new GIFs and copy to organised ablation output folder
        new_gifs = sorted(set(tag_dir.rglob("*.gif")) - before, key=lambda p: p.stat().st_mtime)
        dest.mkdir(parents=True, exist_ok=True)
        for g in new_gifs:
            shutil.copy2(g, dest / g.name)
        print(f"  Done in {elapsed:.0f}s — saved {len(new_gifs)} GIF(s) to {dest}")

print("\nAblation inference complete.")


In [ ]:
# @title Ablation — Step 3: Side-by-Side GIF Comparison
import base64
from pathlib import Path
from IPython.display import display, HTML
from diffsplat_tools.constants import MODEL_SPECS

ABLATION_PROMPTS = [
    "a toy robot",
    "a red sports car",
    "a wooden chair",
    "a tropical plant in a pot",
    "a birthday cake with candles",
]
ABLATION_OUT = DRIVE_BASE / "ablation_outputs"

def prompt_slug(p: str) -> str:
    return p.replace(" ", "_")[:40]

def gif_to_data_uri(path: Path) -> str:
    data = path.read_bytes()
    b64  = base64.b64encode(data).decode()
    return f"data:image/gif;base64,{b64}"

models = ["sd15", "pas", "sd35m"]
labels = {"sd15": "SD 1.5", "pas": "PixArt-Σ", "sd35m": "SD 3.5 M"}

rows = ["<tr><th>Prompt</th>" + "".join(f"<th>{labels[m]}</th>" for m in models) + "</tr>"]
for prompt in ABLATION_PROMPTS:
    slug = prompt_slug(prompt)
    cells = [f"<td style='padding:6px;font-size:12px'>{prompt}</td>"]
    for model_name in models:
        dest = ABLATION_OUT / model_name / slug
        gifs = sorted(dest.glob("*.gif"), key=lambda p: p.stat().st_mtime) if dest.exists() else []
        if gifs:
            uri = gif_to_data_uri(gifs[-1])
            cells.append(f"<td style='padding:4px'><img src=\"{uri}\" width=200></td>")
        else:
            cells.append("<td style='padding:4px;color:#888'>no output</td>")
    rows.append("<tr>" + "".join(cells) + "</tr>")

html = (
    "<table style='border-collapse:collapse;font-family:sans-serif'>"
    + "".join(rows)
    + "</table>"
)
display(HTML(html))


In [ ]:
# @title Ablation — Step 4: CLIP Score (ViT-B/32) per Model
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ftfy", "regex",
                "git+https://github.com/openai/CLIP.git"], check=True)

import torch, clip
from pathlib import Path
from PIL import Image

ABLATION_PROMPTS = [
    "a toy robot",
    "a red sports car",
    "a wooden chair",
    "a tropical plant in a pot",
    "a birthday cake with candles",
]
ABLATION_OUT = DRIVE_BASE / "ablation_outputs"

def prompt_slug(p: str) -> str:
    return p.replace(" ", "_")[:40]

device = "cuda" if torch.cuda.is_available() else "cpu"
model_clip, preprocess = clip.load("ViT-B/32", device=device)
model_clip.eval()

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}
clip_results: dict[str, dict[str, float]] = {}   # model -> prompt -> score

for model_name in ["sd15", "pas", "sd35m"]:
    clip_results[model_name] = {}
    for prompt in ABLATION_PROMPTS:
        slug  = prompt_slug(prompt)
        dest  = ABLATION_OUT / model_name / slug
        imgs  = sorted(
            [p for p in dest.rglob("*") if p.suffix.lower() in IMAGE_EXTS],
            key=lambda p: p.stat().st_mtime,
        ) if dest.exists() else []

        if not imgs:
            clip_results[model_name][prompt] = float("nan")
            continue

        scores = []
        text_tok = clip.tokenize([prompt]).to(device)
        with torch.no_grad():
            text_feat = model_clip.encode_text(text_tok)
            text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
            for img_path in imgs[:8]:   # cap at 8 views per prompt
                try:
                    img = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
                    img_feat = model_clip.encode_image(img)
                    img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
                    scores.append(float((img_feat @ text_feat.T).squeeze()))
                except Exception:
                    pass
        clip_results[model_name][prompt] = sum(scores) / len(scores) if scores else float("nan")
        print(f"[{model_name}] '{prompt}': CLIP = {clip_results[model_name][prompt]:.4f}")

# Store for next cell
import json
(ABLATION_OUT / "clip_scores.json").write_text(json.dumps(clip_results, indent=2))
print("\nCLIP scores saved.")


In [ ]:
# @title Ablation — Step 5: Summary Table
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

ABLATION_OUT = DRIVE_BASE / "ablation_outputs"
score_file   = ABLATION_OUT / "clip_scores.json"

if not score_file.exists():
    print("Run Step 4 (CLIP scoring) first.")
else:
    clip_results = json.loads(score_file.read_text())
    ABLATION_PROMPTS = [
        "a toy robot",
        "a red sports car",
        "a wooden chair",
        "a tropical plant in a pot",
        "a birthday cake with candles",
    ]
    models   = ["sd15", "pas", "sd35m"]
    labels   = {"sd15": "SD 1.5", "pas": "PixArt-Σ", "sd35m": "SD 3.5 M"}

    # Per-prompt breakdown
    rows = []
    for prompt in ABLATION_PROMPTS:
        row = {"Prompt": prompt}
        for m in models:
            row[labels[m]] = round(clip_results.get(m, {}).get(prompt, float("nan")), 4)
        rows.append(row)

    # Mean row
    mean_row = {"Prompt": "**Mean**"}
    for m in models:
        vals = [v for v in clip_results.get(m, {}).values() if v == v]  # filter nan
        mean_row[labels[m]] = round(sum(vals) / len(vals), 4) if vals else float("nan")
    rows.append(mean_row)

    df = pd.DataFrame(rows).set_index("Prompt")

    # Highlight best score per prompt row (excluding mean)
    def highlight_best(row):
        best = row.max()
        return ["font-weight:bold; background:#d4edda" if v == best else "" for v in row]

    styled = (
        df.style
          .apply(highlight_best, axis=1)
          .format("{:.4f}")
          .set_caption("CLIP similarity (ViT-B/32) — higher is better; best per row bolded")
    )
    display(styled)

    # Save CSV
    csv_path = ABLATION_OUT / "ablation_summary.csv"
    df.to_csv(csv_path)
    print(f"\nSaved to {csv_path}")


## Step 11 — (Optional) Evaluate Image Metrics

Compute PSNR, SSIM, and LPIPS between predicted renderings and ground-truth images.

In [ ]:
# @title Evaluation settings
PRED_DIR = ""  # @param {type:"string"} path to predicted images
GT_DIR   = ""  # @param {type:"string"} path to ground-truth images
SKIP_LPIPS = False  # @param {type:"boolean"}

In [ ]:
import json, sys
from pathlib import Path

if not PRED_DIR or not GT_DIR:
    print("Set PRED_DIR and GT_DIR in the cell above to run evaluation.")
else:
    sys.path.insert(0, str(WRAPPER_DIR))
    from diffsplat_tools.evaluation import evaluate_image_dirs
    import argparse

    args = argparse.Namespace(
        pred_dir=PRED_DIR,
        gt_dir=GT_DIR,
        skip_lpips=SKIP_LPIPS,
        device="auto",
        limit=None,
        save_per_image=True,
        json=str(OUT_DIR / "metrics.json"),
    )

    evaluate_image_dirs(args)

## Step 12 — (Optional) Download T3Bench Prompts

In [ ]:
import sys
sys.path.insert(0, str(WRAPPER_DIR))
from diffsplat_tools.downloads import download_t3bench

download_t3bench(DATA_DIR, dry_run=False)

t3bench_file = DATA_DIR / "t3bench" / "t3bench_prompt.txt"
if t3bench_file.exists():
    prompts = t3bench_file.read_text().strip().splitlines()
    print(f"T3Bench loaded: {len(prompts)} prompts")
    print("First 5:", prompts[:5])

---

## Appendix — Drive Storage Layout

```
MyDrive/DiffSplat/
├── pip_cache/           # pip wheel downloads (avoids re-downloading packages)
├── built_wheels/        # pre-compiled CUDA extension wheels
├── hf_home/             # HuggingFace model cache
├── torch_home/          # PyTorch hub cache
├── DiffSplat/           # official DiffSplat repo (chenguolin/DiffSplat)
├── diffsplat-wrapper/   # this wrapper repo (kaustubh484/diffsplat)
├── checkpoints/         # downloaded model weights
│   ├── gsdiff_gobj83k_sd15__render/
│   └── gsdiff_gobj83k_sd15_image__render/
├── data/
│   ├── t3bench/
│   └── gso_rendered/
├── outputs/
│   ├── text/
│   └── image/
└── logs/
```

## Troubleshooting

| Problem | Fix |
|---|---|
| `No GPU found` | `Runtime > Change runtime type > GPU` |
| `ImportError: diffsplat_tools` | Re-run Step 6 (sys.path setup) |
| `xformers` version mismatch | Delete the marker file and re-run Step 2 |
| CUDA out of memory | Use `sd15` model with `HALF_PRECISION=True` |
| Checkpoint download fails | Check HuggingFace token or try again later |
| Rasterizer build fails | Check CUDA toolkit version; try skipping rasterizer |

To force a full reinstall, delete all `.packages_installed_*` and `.reqs_installed_*` marker files from `MyDrive/DiffSplat/`.